# FREIGHT-MNet: Calibration and Decision-Aware CVaR Evaluation

This notebook performs the next evaluation stage after the stabilized D-CQHGT disruption-gate experiment.

It has three concrete goals:

1. **Repair sparse-segment diagnostics and report generation.**  
   The previous segment summary failed when `n_fmi_county_pairs` was not present or was renamed during merges.  
   This notebook rebuilds sparse-support segment labels from the supervised OD-year table and never depends on `DataFrame.to_markdown()`.

2. **Calibrate quantile predictions.**  
   The notebook applies validation-year residual calibration to existing val/test predictions and evaluates whether calibration improves 2024 Cold-OD prediction.

3. **Run decision-aware CVaR-style evaluation.**  
   Because the current benchmark has OD-year labels rather than full route-level labels, this notebook implements an OD-risk screening decision proxy: models rank Cold-OD test lanes by predicted CVaR-like risk, and the evaluation measures how much realized tail risk the selected top-risk lanes capture compared with the oracle top-risk set.

This notebook does **not** retrain graph models. It consumes previously saved prediction artifacts and produces repaired diagnostics, calibration metrics, and decision-aware evaluation tables.

## 1. Kernel preflight

This notebook only needs Pandas/NumPy/Matplotlib, not PyTorch Geometric. However, the project has repeatedly shown that the base Anaconda kernel can mix incompatible NumPy and compiled packages. This preflight cell prints the active Python executable and warns if it appears to be the unstable base environment.

If the notebook later fails during Pandas import, switch to the clean project kernel, such as:

```text
Python (freight_mnet_cuda)
```

or the clean analysis environment used for prior experiment notebooks.

In [2]:
# This cell intentionally avoids importing pandas or numpy.
import os
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Current working directory:", os.getcwd())
EXPECTED_PROJECT_ENV_FRAGMENT = r"E:\NetworkOptimization\pythonProject1\env"
if EXPECTED_PROJECT_ENV_FRAGMENT.lower() not in sys.executable.lower():
    print("\nWarning:")
    print("  The active kernel does not appear to be the project virtual environment.")
    print("  If pandas/pyarrow/numpy import errors occur, switch to the project kernel.")
    print("  Expected path fragment:", EXPECTED_PROJECT_ENV_FRAGMENT)

Python executable: E:\NetworkOptimization\pythonProject1\env\.venv_freight_mnet_cuda\Scripts\python.exe
Python version: 3.11.5 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:26:23) [MSC v.1916 64 bit (AMD64)]
Current working directory: E:\NetworkOptimization\pythonProject1\Code


## 2. Imports and experiment configuration

The configuration below centralizes all paths and experiment choices. The default paths follow the project folder layout used in earlier notebooks.

In [3]:
from __future__ import annotations

import json
import math
import os
import re
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

# Disable optional pandas acceleration backends. This reduces the chance of
# optional numexpr/bottleneck ABI issues in mixed package environments.
pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib available:", MATPLOTLIB_AVAILABLE)


@dataclass
class EvalConfig:
    """Configuration for calibration and decision-aware evaluation."""
    data_root: Path = Path(r"E:\NetworkOptimization\pythonProject1\Data")
    scope: str = "east_plus_gulf"
    run_name: str = "calibration_cvar_evaluation_v1_notebook"

    # Main prediction artifact from the previous stabilized disruption-gate run.
    disruption_stabilization_run: str = "dcqhgt_disruption_gate_stabilization_v1_notebook"

    # Reference artifacts from earlier experiments. These are optional if the
    # combined stabilized prediction file already contains references.
    graphv2_run: str = "graphsage_hgt_cold_od_baselines_v2_notebook"
    coldod_run: str = "cold_od_split_baselines_v1_notebook"

    expected_test_rows: int = 1057

    # Calibration settings.
    calibration_bins: int = 10
    min_val_rows_for_group_calibration: int = 100
    min_bin_rows: int = 25
    calibration_methods: Tuple[str, ...] = (
        "none",
        "global_median",
        "pred_q75_decile",
        "pred_iqr_decile",
    )

    # Decision-aware risk-screening settings.
    cvar_eta: float = 0.50
    topk_fractions: Tuple[float, ...] = (0.05, 0.10, 0.20)

    # Output controls.
    overwrite: bool = True
    create_plots: bool = True


cfg = EvalConfig()

# Core paths.
scope_dir = cfg.scope
experiment_root = cfg.data_root / "10_experiments"

output_dir = experiment_root / cfg.run_name / scope_dir
tables_dir = output_dir / "tables"
plots_dir = output_dir / "plots"
reports_dir = output_dir / "reports"

for directory in [output_dir, tables_dir, plots_dir, reports_dir]:
    directory.mkdir(parents=True, exist_ok=True)

# Supervised tables.
supervised_disruption_path = (
    cfg.data_root
    / "08_processed"
    / "disruption_features"
    / f"freight_mnet_supervised_edges_2018_2024_{cfg.scope}_with_disruption.parquet"
)
supervised_model_ready_path = (
    cfg.data_root
    / "08_processed"
    / "model_ready"
    / f"freight_mnet_supervised_edges_2018_2024_{cfg.scope}.parquet"
)

# Prediction artifacts.
disruption_predictions_path = (
    experiment_root
    / cfg.disruption_stabilization_run
    / scope_dir
    / "combined_predictions_disruption_gate_stabilized.parquet"
)
disruption_seed_ensemble_path = (
    experiment_root
    / cfg.disruption_stabilization_run
    / scope_dir
    / "predictions_disruption_gate_stabilized_seed_ensemble.parquet"
)
graphv2_predictions_path = (
    experiment_root
    / cfg.graphv2_run
    / scope_dir
    / "combined_predictions_graph_cold_od_val_test_v2.parquet"
)
coldod_predictions_path = (
    experiment_root
    / cfg.coldod_run
    / scope_dir
    / "predictions_cold_od_val_test.parquet"
)

print("Output directory:", output_dir)
print("Primary predictions path:", disruption_predictions_path)

NumPy: 2.4.5
Pandas: 2.3.3
Matplotlib available: True
Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\calibration_cvar_evaluation_v1_notebook\east_plus_gulf
Primary predictions path: E:\NetworkOptimization\pythonProject1\Data\10_experiments\dcqhgt_disruption_gate_stabilization_v1_notebook\east_plus_gulf\combined_predictions_disruption_gate_stabilized.parquet


## 3. Utility functions

These helpers handle robust file checks, safe table output, Parquet dtype normalization, and text-table reporting without requiring the optional `tabulate` package.

In [4]:
def ensure_file_exists(path: Path, description: str) -> None:
    """Raise a clear error if a required artifact is missing."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{description} was not found: {path}")


def print_frame(title: str, frame: pd.DataFrame, max_rows: int = 20) -> None:
    """Print a compact dataframe preview without using tabulate."""
    print(f"\n{title}")
    if frame.empty:
        print("(empty)")
    else:
        print(frame.head(max_rows).to_string(index=False))


def dataframe_to_text_table(
    frame: pd.DataFrame,
    columns: Optional[Sequence[str]] = None,
    max_rows: int = 20,
) -> str:
    """
    Convert a dataframe preview to a plain-text table.

    This avoids pandas.DataFrame.to_markdown(), which requires the optional
    tabulate dependency and previously caused report-generation failures.
    """
    if frame is None or frame.empty:
        return "_No rows available._"

    if columns is None:
        cols = list(frame.columns)
    else:
        cols = [col for col in columns if col in frame.columns]

    if not cols:
        return "_No requested columns are available._"

    return "```text\n" + frame[cols].head(max_rows).to_string(index=False) + "\n```"


def json_default(value):
    """Serialize numpy, pandas, and pathlib values for JSON output."""
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    return str(value)


def normalize_object_value(value):
    """Convert mixed object values to Parquet-safe strings or missing values."""
    if value is None:
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except Exception:
        pass
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, default=json_default, sort_keys=True)
    return str(value)


def normalize_dataframe_for_parquet(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with object-like columns converted to string dtype."""
    clean = frame.copy()
    clean.columns = [str(col) for col in clean.columns]

    for col in clean.columns:
        series = clean[col]
        if (
            pd.api.types.is_object_dtype(series)
            or pd.api.types.is_string_dtype(series)
            or str(series.dtype) == "category"
        ):
            clean[col] = series.map(normalize_object_value).astype("string")

    return clean


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> None:
    """Write a dataframe to Parquet after object-dtype normalization."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    normalize_dataframe_for_parquet(frame).to_parquet(path, index=False)


def safe_write_json(payload: dict, path: Path) -> None:
    """Write a JSON artifact using robust serialization."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=json_default)

## 4. Load supervised OD-year lookup

Calibration, sparse segmentation, and paired evaluation all require reliable `row_id`, true labels, `n_fmi_county_pairs`, and sample weights. This cell builds a compact lookup from the supervised OD-year table.

In [5]:
def find_supervised_path() -> Path:
    """Choose the supervised table path to use for lookup metadata."""
    if supervised_disruption_path.exists():
        return supervised_disruption_path
    if supervised_model_ready_path.exists():
        return supervised_model_ready_path
    raise FileNotFoundError(
        "No supervised table was found. Checked:\n"
        f"  {supervised_disruption_path}\n"
        f"  {supervised_model_ready_path}"
    )


def normalize_faf_code(value: object) -> Optional[str]:
    """Normalize a FAF zone code into a three-character string."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = str(value).strip()
    if not text:
        return None

    try:
        numeric = float(text)
        if np.isfinite(numeric):
            return str(int(numeric)).zfill(3)
    except Exception:
        pass

    digits = "".join(ch for ch in text if ch.isdigit())
    if not digits:
        return None
    return digits[-3:].zfill(3)


def find_first_column(columns: Sequence[str], candidates: Sequence[str]) -> Optional[str]:
    """Return the first candidate column that exists."""
    col_set = set(columns)
    for candidate in candidates:
        if candidate in col_set:
            return candidate
    return None


ORIGIN_CANDIDATES = [
    "faf_orig_str", "faf_orig", "orig_faf", "origin_faf", "origin", "orig", "from_faf", "o_faf"
]
DEST_CANDIDATES = [
    "faf_dest_str", "faf_dest", "dest_faf", "destination_faf", "destination", "dest", "to_faf", "d_faf"
]
YEAR_CANDIDATES = ["year_int", "year", "YEAR", "forecast_year", "t"]


def build_supervised_lookup(supervised_path: Path) -> pd.DataFrame:
    """Build a row-level metadata lookup from the supervised table."""
    df = pd.read_parquet(supervised_path).copy()

    if "row_id" not in df.columns:
        df["row_id"] = np.arange(len(df), dtype=np.int64)

    origin_col = find_first_column(df.columns, ORIGIN_CANDIDATES)
    dest_col = find_first_column(df.columns, DEST_CANDIDATES)
    year_col = find_first_column(df.columns, YEAR_CANDIDATES)

    if origin_col is None or dest_col is None or year_col is None:
        raise ValueError(
            "Could not identify origin/destination/year columns in supervised table. "
            f"Available columns: {list(df.columns)}"
        )

    df["faf_orig_str"] = df[origin_col].map(normalize_faf_code)
    df["faf_dest_str"] = df[dest_col].map(normalize_faf_code)
    df["year_int"] = pd.to_numeric(df[year_col], errors="raise").astype(int)

    label_map = {
        "true_q25": ["true_q25", "truck_q25"],
        "true_q50": ["true_q50", "truck_q50"],
        "true_q75": ["true_q75", "truck_q75"],
    }
    for target, candidates in label_map.items():
        src_col = find_first_column(df.columns, candidates)
        if src_col is None:
            raise ValueError(f"Missing label column for {target}.")
        df[target] = pd.to_numeric(df[src_col], errors="raise")

    if "sample_weight" not in df.columns:
        if "obs_weight_sum" in df.columns:
            df["sample_weight"] = pd.to_numeric(df["obs_weight_sum"], errors="coerce").fillna(1.0)
        else:
            df["sample_weight"] = 1.0

    if "n_fmi_county_pairs" not in df.columns:
        df["n_fmi_county_pairs"] = np.nan

    if "cold_split" not in df.columns:
        if "year_int" in df.columns:
            df["cold_split"] = np.where(df["year_int"].eq(2024), "test", np.where(df["year_int"].eq(2023), "val", "train"))
        else:
            df["cold_split"] = "unknown"

    lookup_cols = [
        "row_id",
        "faf_orig_str",
        "faf_dest_str",
        "year_int",
        "cold_split",
        "true_q25",
        "true_q50",
        "true_q75",
        "sample_weight",
        "n_fmi_county_pairs",
    ]

    lookup = df[lookup_cols].copy()
    lookup["row_id"] = pd.to_numeric(lookup["row_id"], errors="raise").astype(int)
    lookup["sample_weight"] = pd.to_numeric(lookup["sample_weight"], errors="coerce").fillna(1.0)
    lookup["n_fmi_county_pairs"] = pd.to_numeric(lookup["n_fmi_county_pairs"], errors="coerce")

    if lookup["row_id"].duplicated().any():
        raise ValueError("Supervised lookup has duplicated row_id values.")

    return lookup


supervised_path = find_supervised_path()
supervised_lookup = build_supervised_lookup(supervised_path)

print("Using supervised path:", supervised_path)
print("Supervised lookup shape:", supervised_lookup.shape)
print_frame("Supervised lookup preview", supervised_lookup, max_rows=5)
print("Cold split counts:")
print(supervised_lookup["cold_split"].value_counts(dropna=False).to_string())

Using supervised path: E:\NetworkOptimization\pythonProject1\Data\08_processed\disruption_features\freight_mnet_supervised_edges_2018_2024_east_plus_gulf_with_disruption.parquet
Supervised lookup shape: (73972, 10)

Supervised lookup preview
 row_id faf_orig_str faf_dest_str  year_int cold_split  true_q25  true_q50  true_q75  sample_weight  n_fmi_county_pairs
      0          011          011      2018      train      1.00     37.02     71.28    1541.730999                 110
      1          011          012      2018      train    248.18    943.27   1925.53      56.596443                  22
      2          011          019      2018      train     48.00    100.10    218.95    3706.180871                 558
      3          011          050      2018      train    404.01   1010.39   1818.68    1365.823515                 433
      4          011          091      2018      train   2579.38   3718.82   5127.33      15.000000                  15
Cold split counts:
cold_split
train   

## 5. Load and normalize prediction artifacts

This cell loads the stabilized disruption predictions and optional reference artifacts. It reconstructs missing `row_id` values when necessary and attaches true labels, sample weights, and sparse-support metadata.

In [6]:
def add_od_year_merge_keys(frame: pd.DataFrame) -> pd.DataFrame:
    """Add normalized OD-year merge keys to a prediction dataframe."""
    clean = frame.copy()

    origin_col = find_first_column(clean.columns, ORIGIN_CANDIDATES)
    dest_col = find_first_column(clean.columns, DEST_CANDIDATES)
    year_col = find_first_column(clean.columns, YEAR_CANDIDATES)

    if origin_col is None or dest_col is None or year_col is None:
        raise ValueError(
            "Cannot reconstruct row_id because origin/destination/year columns are missing. "
            f"Available columns: {list(clean.columns)}"
        )

    clean["faf_orig_str"] = clean[origin_col].map(normalize_faf_code)
    clean["faf_dest_str"] = clean[dest_col].map(normalize_faf_code)
    clean["year_int"] = pd.to_numeric(clean[year_col], errors="raise").astype(int)
    return clean


def normalize_prediction_schema(frame: pd.DataFrame, default_source: str) -> pd.DataFrame:
    """Normalize prediction columns and attach row-level metadata from supervised_lookup."""
    clean = frame.copy()

    rename = {
        "actual_q25": "true_q25",
        "actual_q50": "true_q50",
        "actual_q75": "true_q75",
        "y_q25": "true_q25",
        "y_q50": "true_q50",
        "y_q75": "true_q75",
        "truck_q25": "true_q25",
        "truck_q50": "true_q50",
        "truck_q75": "true_q75",
        "prediction_q25": "pred_q25",
        "prediction_q50": "pred_q50",
        "prediction_q75": "pred_q75",
        "q25_pred": "pred_q25",
        "q50_pred": "pred_q50",
        "q75_pred": "pred_q75",
        "predicted_q25": "pred_q25",
        "predicted_q50": "pred_q50",
        "predicted_q75": "pred_q75",
        "q25_hat": "pred_q25",
        "q50_hat": "pred_q50",
        "q75_hat": "pred_q75",
        "cold_split": "split",
        "temporal_split": "split",
        "dataset_split": "split",
        "rowid": "row_id",
        "row_idx": "row_id",
        "edge_row_id": "row_id",
        "supervised_row_id": "row_id",
    }
    clean = clean.rename(columns={k: v for k, v in rename.items() if k in clean.columns})

    if "source" not in clean.columns:
        clean["source"] = default_source
    if "model" not in clean.columns:
        clean["model"] = "unknown"
    if "checkpoint_metric" not in clean.columns:
        if "checkpoint" in clean.columns:
            clean["checkpoint_metric"] = clean["checkpoint"]
        elif "variant" in clean.columns:
            clean["checkpoint_metric"] = clean["variant"]
        else:
            clean["checkpoint_metric"] = "baseline"
    if "seed" not in clean.columns:
        clean["seed"] = "baseline"
    if "topology_candidate" not in clean.columns:
        clean["topology_candidate"] = clean["model"].astype(str)
    if "disruption_group" not in clean.columns:
        clean["disruption_group"] = "baseline"
    if "split" not in clean.columns:
        clean["split"] = "unknown"

    if "row_id" not in clean.columns:
        keyed = add_od_year_merge_keys(clean)
        clean = keyed.merge(
            supervised_lookup[
                [
                    "row_id",
                    "faf_orig_str",
                    "faf_dest_str",
                    "year_int",
                    "true_q25",
                    "true_q50",
                    "true_q75",
                    "sample_weight",
                    "n_fmi_county_pairs",
                ]
            ],
            on=["faf_orig_str", "faf_dest_str", "year_int"],
            how="left",
            validate="many_to_one",
            suffixes=("", "_lookup"),
        )
        if clean["row_id"].isna().any():
            bad = clean.loc[clean["row_id"].isna()].head(5)
            raise ValueError(
                "Failed to reconstruct row_id for some predictions. "
                f"Example rows:\n{bad.to_string(index=False)}"
            )
    else:
        clean["row_id"] = pd.to_numeric(clean["row_id"], errors="raise").astype(int)
        merge_cols = ["row_id"]
        missing_metadata_cols = [
            col for col in ["true_q25", "true_q50", "true_q75", "sample_weight", "n_fmi_county_pairs"]
            if col not in clean.columns
        ]
        if missing_metadata_cols:
            clean = clean.merge(
                supervised_lookup[merge_cols + missing_metadata_cols],
                on="row_id",
                how="left",
                validate="many_to_one",
            )

    # Coalesce possible duplicated metadata columns.
    for base_col in ["true_q25", "true_q50", "true_q75", "sample_weight", "n_fmi_county_pairs"]:
        candidates = [base_col, f"{base_col}_x", f"{base_col}_y", f"{base_col}_lookup"]
        available = [col for col in candidates if col in clean.columns]
        if available:
            value = clean[available[0]]
            for col in available[1:]:
                value = value.combine_first(clean[col])
            clean[base_col] = value
            clean = clean.drop(columns=[col for col in available if col != base_col], errors="ignore")

    required = ["split", "row_id", "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75"]
    missing = [col for col in required if col not in clean.columns]
    if missing:
        print("Available columns after normalization:")
        print(list(clean.columns))
        raise ValueError(f"Prediction schema missing required columns: {missing}")

    # Standardize types.
    clean["row_id"] = pd.to_numeric(clean["row_id"], errors="raise").astype(int)
    clean["split"] = clean["split"].astype(str).str.lower()
    for col in ["source", "model", "topology_candidate", "disruption_group", "checkpoint_metric", "seed"]:
        clean[col] = clean[col].astype(str)

    for col in ["true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75", "sample_weight", "n_fmi_county_pairs"]:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

    clean["sample_weight"] = clean["sample_weight"].fillna(1.0)

    return clean


prediction_frames = []

if disruption_predictions_path.exists():
    prediction_frames.append(normalize_prediction_schema(pd.read_parquet(disruption_predictions_path), "DISRUPTION_STABILIZED"))
elif disruption_seed_ensemble_path.exists():
    prediction_frames.append(normalize_prediction_schema(pd.read_parquet(disruption_seed_ensemble_path), "DISRUPTION_STABILIZED_ENSEMBLE"))
else:
    raise FileNotFoundError(
        "No stabilized disruption prediction artifact was found. Checked:\n"
        f"  {disruption_predictions_path}\n"
        f"  {disruption_seed_ensemble_path}"
    )

# Load references if available. Duplicates are handled downstream.
if graphv2_predictions_path.exists():
    prediction_frames.append(normalize_prediction_schema(pd.read_parquet(graphv2_predictions_path), "GraphV2"))

if coldod_predictions_path.exists():
    prediction_frames.append(normalize_prediction_schema(pd.read_parquet(coldod_predictions_path), "ColdOD"))

raw_predictions = pd.concat(prediction_frames, ignore_index=True)
print("Raw normalized prediction rows:", raw_predictions.shape)
print_frame("Prediction preview", raw_predictions, max_rows=5)

C:\Users\Nick_James\AppData\Local\Temp\ipykernel_83824\2678867336.py:126: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  value = value.combine_first(clean[col])
C:\Users\Nick_James\AppData\Local\Temp\ipykernel_83824\2678867336.py:126: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  value = value.combine_first(clean[col])
C:\Users\Nick_James\AppData\Local\Temp\ipykernel_83824\2678867336.py:126: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the

Raw normalized prediction rows: (136095, 39)

Prediction preview
   source                     model       variant split faf_orig faf_dest  od_pair   year  true_q25  true_q50  true_q75    pred_q25    pred_q50    pred_q75  raw_crossing  sample_weight  n_fmi_county_pairs  obs_weight_sum  true_iqr    pred_iqr  abs_error_q25  abs_error_q50  abs_error_q75  abs_error_iqr  pinball_q25  pinball_q50  pinball_q75  pinball_mean_row      checkpoint_metric      seed disruption_group        topology_candidate  row_id  error_q25  error_q50  error_q75 faf_orig_str faf_dest_str  year_int
REFERENCE coldod_mlp_residual_prior postprocessed  test      011      123 011__123 2024.0    956.93   1384.72   2500.75  848.514343 1495.657471 2718.356934           0.0       1.173914                77.0      681.565142   1543.82 1869.842590     108.415657     110.937471     217.606934     326.022590    27.103914    55.468735    54.401733         45.658128 deduplicated_reference reference        reference coldod_mlp_r

## 6. Deduplicate model groups and attach strict test-only segments

This cell fixes the sparse-segment issue by building all segment labels from a unique row-level lookup, then merging those labels onto every model prediction. This prevents the previous `n_fmi_county_pairs` KeyError and ensures every model group sums to exactly the expected test-row count.

In [7]:
MODEL_ID_COLS = [
    "source",
    "model",
    "topology_candidate",
    "disruption_group",
    "checkpoint_metric",
    "seed",
]


def add_prediction_error_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Add row-level error, absolute error, IQR, and pinball columns."""
    out = frame.copy()

    for q_col in ["q25", "q50", "q75"]:
        true_col = f"true_{q_col}"
        pred_col = f"pred_{q_col}"
        out[f"error_{q_col}"] = out[pred_col] - out[true_col]
        out[f"abs_error_{q_col}"] = out[f"error_{q_col}"].abs()

    out["true_iqr"] = out["true_q75"] - out["true_q25"]
    out["pred_iqr"] = out["pred_q75"] - out["pred_q25"]
    out["abs_error_iqr"] = (out["pred_iqr"] - out["true_iqr"]).abs()

    for tau, q_col in [(0.25, "q25"), (0.50, "q50"), (0.75, "q75")]:
        y = out[f"true_{q_col}"]
        q = out[f"pred_{q_col}"]
        diff = y - q
        out[f"pinball_{q_col}"] = np.maximum(tau * diff, (tau - 1.0) * diff)

    out["pinball_mean_row"] = out[["pinball_q25", "pinball_q50", "pinball_q75"]].mean(axis=1)

    return out


def deduplicate_prediction_groups(frame: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure each model group has at most one prediction per row_id.

    If duplicates exist, this function keeps the first after a stable sort.
    """
    clean = frame.copy()
    for col in MODEL_ID_COLS:
        if col not in clean.columns:
            clean[col] = "unknown"
        clean[col] = clean[col].astype(str)

    before = len(clean)
    clean = (
        clean.sort_values(MODEL_ID_COLS + ["split", "row_id"])
        .drop_duplicates(MODEL_ID_COLS + ["row_id"], keep="first")
        .reset_index(drop=True)
    )
    after = len(clean)
    if before != after:
        print(f"Deduplicated prediction rows: {before:,} -> {after:,}")

    return clean


def build_test_segment_lookup(supervised_lookup: pd.DataFrame) -> pd.DataFrame:
    """Create strict test-only segment labels from the supervised lookup."""
    lookup = supervised_lookup.copy()
    lookup["split"] = lookup["cold_split"].astype(str).str.lower()
    test = lookup.loc[lookup["split"].eq("test")].copy()

    if test.empty:
        raise ValueError("No test rows found in supervised lookup.")

    test["true_iqr"] = test["true_q75"] - test["true_q25"]

    def qbin(series: pd.Series, q: int, prefix: str) -> pd.Series:
        values = pd.to_numeric(series, errors="coerce")
        if values.notna().sum() == 0 or values.nunique(dropna=True) < 2:
            return pd.Series([f"{prefix}_01"] * len(values), index=values.index, dtype="object")
        codes = pd.qcut(values, q=q, labels=False, duplicates="drop")
        return pd.Series(
            [f"{prefix}_{int(code) + 1:02d}" if pd.notna(code) else "missing" for code in codes],
            index=values.index,
            dtype="object",
        )

    test["true_q75_decile"] = qbin(test["true_q75"], 10, "true_q75_decile")
    test["true_iqr_decile"] = qbin(test["true_iqr"], 10, "true_iqr_decile")

    if test["n_fmi_county_pairs"].notna().sum() > 0 and test["n_fmi_county_pairs"].nunique(dropna=True) > 1:
        test["n_fmi_county_pairs_quartile"] = qbin(test["n_fmi_county_pairs"], 4, "n_fmi_q")
    else:
        test["n_fmi_county_pairs_quartile"] = "missing"

    return test[
        [
            "row_id",
            "true_iqr",
            "n_fmi_county_pairs",
            "true_q75_decile",
            "true_iqr_decile",
            "n_fmi_county_pairs_quartile",
        ]
    ].copy()


def attach_segments(frame: pd.DataFrame, segment_lookup: pd.DataFrame) -> pd.DataFrame:
    """Attach strict test-only segment labels to prediction rows."""
    out = frame.drop(
        columns=[
            "true_iqr",
            "n_fmi_county_pairs",
            "true_q75_decile",
            "true_iqr_decile",
            "n_fmi_county_pairs_quartile",
        ],
        errors="ignore",
    ).merge(segment_lookup, on="row_id", how="left", validate="many_to_one")

    for col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
        out[col] = out[col].fillna("not_test")

    return out


predictions = deduplicate_prediction_groups(raw_predictions)
predictions = add_prediction_error_columns(predictions)

test_segment_lookup = build_test_segment_lookup(supervised_lookup)
predictions = attach_segments(predictions, test_segment_lookup)

safe_to_parquet(predictions, output_dir / "normalized_predictions_with_repaired_segments.parquet")

print("Predictions with repaired segments:", predictions.shape)
print("Unique test row_ids in segment lookup:", test_segment_lookup["row_id"].nunique())
print_frame("Segment lookup preview", test_segment_lookup, max_rows=8)

Predictions with repaired segments: (136095, 42)
Unique test row_ids in segment lookup: 10574

Segment lookup preview
 row_id  true_iqr  n_fmi_county_pairs    true_q75_decile    true_iqr_decile n_fmi_county_pairs_quartile
  63398    145.98                 110 true_q75_decile_01 true_iqr_decile_01                  n_fmi_q_03
  63399   1968.05                  22 true_q75_decile_03 true_iqr_decile_04                  n_fmi_q_01
  63400    694.37                 594 true_q75_decile_01 true_iqr_decile_01                  n_fmi_q_04
  63401   1418.11                 652 true_q75_decile_02 true_iqr_decile_03                  n_fmi_q_04
  63402   3012.30                  16 true_q75_decile_08 true_iqr_decile_08                  n_fmi_q_01
  63403   3302.27                  33 true_q75_decile_08 true_iqr_decile_09                  n_fmi_q_02
  63404   2938.45                  16 true_q75_decile_08 true_iqr_decile_08                  n_fmi_q_01
  63405   3140.50                  17 true_q75_dec

## 7. Metric functions

Metrics are computed at the prediction-row level and then aggregated by model group, calibration method, and decision-evaluation scope.

In [8]:
TAUS = np.array([0.25, 0.50, 0.75], dtype=float)


def pinball_loss_array(true: np.ndarray, pred: np.ndarray) -> np.ndarray:
    """Return element-wise pinball losses for q25/q50/q75."""
    diff = true - pred
    return np.maximum(TAUS * diff, (TAUS - 1.0) * diff)


def compute_metrics_from_arrays(
    true: np.ndarray,
    pred: np.ndarray,
    weights: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    """Compute quantile prediction metrics."""
    true = np.asarray(true, dtype=float)
    pred = np.asarray(pred, dtype=float)

    if weights is None:
        weights = np.ones(true.shape[0], dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)
        weights = np.where(np.isfinite(weights), weights, 1.0)
        weights = np.maximum(weights, 0.0)

    losses = pinball_loss_array(true, pred)
    abs_error = np.abs(pred - true)
    true_iqr = true[:, 2] - true[:, 0]
    pred_iqr = pred[:, 2] - pred[:, 0]

    metric = {
        "n_rows": int(true.shape[0]),
        "pinball_q25": float(np.mean(losses[:, 0])),
        "pinball_q50": float(np.mean(losses[:, 1])),
        "pinball_q75": float(np.mean(losses[:, 2])),
        "pinball_mean": float(np.mean(losses)),
        "weighted_pinball_mean": float(np.average(losses.mean(axis=1), weights=weights)),
        "mae_q25": float(np.mean(abs_error[:, 0])),
        "mae_q50": float(np.mean(abs_error[:, 1])),
        "mae_q75": float(np.mean(abs_error[:, 2])),
        "iqr_mae": float(np.mean(np.abs(pred_iqr - true_iqr))),
        "bias_q25": float(np.mean(pred[:, 0] - true[:, 0])),
        "bias_q50": float(np.mean(pred[:, 1] - true[:, 1])),
        "bias_q75": float(np.mean(pred[:, 2] - true[:, 2])),
        "pred_iqr_mean": float(np.mean(pred_iqr)),
        "true_iqr_mean": float(np.mean(true_iqr)),
        "raw_crossing_rate": float(np.mean((pred[:, 0] > pred[:, 1]) | (pred[:, 1] > pred[:, 2]))),
        "q50_inside_q25_q75_rate": float(np.mean((true[:, 1] >= pred[:, 0]) & (true[:, 1] <= pred[:, 2]))),
    }
    return metric


def compute_group_metrics(
    frame: pd.DataFrame,
    group_cols: Sequence[str],
    split_filter: Optional[str] = None,
) -> pd.DataFrame:
    """Compute metrics by group columns."""
    if split_filter is not None:
        data = frame.loc[frame["split"].astype(str).str.lower().eq(split_filter)].copy()
    else:
        data = frame.copy()

    rows = []
    for keys, group in data.groupby(list(group_cols), dropna=False):
        true = group[["true_q25", "true_q50", "true_q75"]].to_numpy(float)
        pred = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
        weights = group["sample_weight"].to_numpy(float) if "sample_weight" in group.columns else None
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        row.update(compute_metrics_from_arrays(true, pred, weights))
        rows.append(row)

    return pd.DataFrame(rows)

## 8. Repaired sparse, stress, and risk-segment summaries

This section regenerates the segment summaries that previously failed or produced missing sparse labels.

In [9]:
def segment_summary(frame: pd.DataFrame, segment_col: str) -> pd.DataFrame:
    """Compute strict test-only segment summaries for a segment column."""
    test = frame.loc[frame["split"].astype(str).str.lower().eq("test")].copy()

    group_cols = MODEL_ID_COLS + [segment_col]
    rows = []

    for keys, group in test.groupby(group_cols, dropna=False):
        true = group[["true_q25", "true_q50", "true_q75"]].to_numpy(float)
        pred = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
        weights = group["sample_weight"].to_numpy(float) if "sample_weight" in group.columns else None
        row = dict(zip(group_cols, keys))
        row.update(compute_metrics_from_arrays(true, pred, weights))
        rows.append(row)

    summary = pd.DataFrame(rows)

    totals = (
        summary.groupby(MODEL_ID_COLS, as_index=False)["n_rows"]
        .sum()
        .rename(columns={"n_rows": "segment_total_rows"})
    )
    bad = totals.loc[totals["segment_total_rows"].ne(cfg.expected_test_rows)]
    if not bad.empty:
        print(f"Warning: some {segment_col} model groups do not sum to {cfg.expected_test_rows}.")
        print(bad.head(10).to_string(index=False))
    else:
        print(f"{segment_col}: all model groups sum to expected test rows = {cfg.expected_test_rows}.")

    return summary


segment_tables = {}
for col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
    table = segment_summary(predictions, col)
    segment_tables[col] = table
    table.to_csv(tables_dir / f"repaired_segment_summary__{col}.csv", index=False)
    print_frame(f"Repaired segment summary: {col}", table, max_rows=10)

sparse_audit = (
    segment_tables["n_fmi_county_pairs_quartile"]
    .groupby(["source", "model", "topology_candidate", "disruption_group", "checkpoint_metric", "seed"], as_index=False)
    .agg(
        n_segments=("n_fmi_county_pairs_quartile", "nunique"),
        total_rows=("n_rows", "sum"),
    )
)
sparse_audit.to_csv(tables_dir / "repaired_sparse_segment_audit.csv", index=False)
print_frame("Sparse segment audit", sparse_audit, max_rows=20)

true_q75_decile: all model groups sum to expected test rows = 1057.

Repaired segment summary: true_q75_decile
source             model topology_candidate disruption_group checkpoint_metric           seed    true_q75_decile  n_rows  pinball_q25  pinball_q50  pinball_q75  pinball_mean  weighted_pinball_mean     mae_q25     mae_q50     mae_q75     iqr_mae     bias_q25     bias_q50     bias_q75  pred_iqr_mean  true_iqr_mean  raw_crossing_rate  q50_inside_q25_q75_rate
ColdOD destination_prior  destination_prior         baseline    not_applicable not_applicable true_q75_decile_01     106   997.290925   917.669064   644.093066    853.017685             850.081578 1329.721233 1835.338129 2576.372265 1246.651031  1329.721233  1835.338129  2576.372265    2091.003108     844.352077                0.0                 0.000000
ColdOD destination_prior  destination_prior         baseline    not_applicable not_applicable true_q75_decile_02     106   665.374430   530.866213   436.374069    544.204904

## 9. Calibration utilities

Calibration is fit on validation rows and then applied to validation and test rows for reporting. The key methods are `none`, `global_median`, `pred_q75_decile`, and `pred_iqr_decile`.

In [10]:
def enforce_monotone_quantiles(pred: np.ndarray) -> np.ndarray:
    """Enforce q25 <= q50 <= q75 by sorting each row."""
    return np.sort(np.asarray(pred, dtype=float), axis=1)


def robust_quantile_edges(values: pd.Series, n_bins: int) -> Optional[np.ndarray]:
    """Build unique quantile edges for binning; return None if impossible."""
    values = pd.to_numeric(values, errors="coerce").dropna().to_numpy(float)
    if values.size < 2 or np.unique(values).size < 2:
        return None
    edges = np.unique(np.quantile(values, np.linspace(0.0, 1.0, n_bins + 1)))
    if edges.size < 3:
        return None
    return edges


def assign_bins_from_edges(values: pd.Series, edges: np.ndarray) -> pd.Series:
    """Assign values to integer bins based on precomputed quantile edges."""
    arr = pd.to_numeric(values, errors="coerce").to_numpy(float)
    bin_ids = np.searchsorted(edges[1:-1], arr, side="right")
    bin_ids = np.where(np.isfinite(arr), bin_ids, -1)
    return pd.Series(bin_ids, index=values.index, dtype="Int64")


def fit_global_median_calibration(val: pd.DataFrame) -> dict:
    """Fit global median residual calibration on validation rows."""
    residuals = {
        "q25": float(np.median(val["true_q25"] - val["pred_q25"])),
        "q50": float(np.median(val["true_q50"] - val["pred_q50"])),
        "q75": float(np.median(val["true_q75"] - val["pred_q75"])),
    }
    return {"type": "global_median", "residuals": residuals}


def fit_binned_median_calibration(
    val: pd.DataFrame,
    bin_feature: str,
    n_bins: int,
    min_bin_rows: int,
) -> dict:
    """Fit bin-level median residual calibration."""
    edges = robust_quantile_edges(val[bin_feature], n_bins)
    global_params = fit_global_median_calibration(val)

    if edges is None:
        return {"type": "binned_median", "bin_feature": bin_feature, "edges": None, "bins": {}, "fallback": global_params}

    work = val.copy()
    work["_calib_bin"] = assign_bins_from_edges(val[bin_feature], edges)

    bin_params = {}
    for bin_id, group in work.groupby("_calib_bin", dropna=False):
        if pd.isna(bin_id) or int(bin_id) < 0 or len(group) < min_bin_rows:
            continue
        bin_params[str(int(bin_id))] = {
            "n_rows": int(len(group)),
            "q25": float(np.median(group["true_q25"] - group["pred_q25"])),
            "q50": float(np.median(group["true_q50"] - group["pred_q50"])),
            "q75": float(np.median(group["true_q75"] - group["pred_q75"])),
        }

    return {
        "type": "binned_median",
        "bin_feature": bin_feature,
        "edges": edges.tolist(),
        "bins": bin_params,
        "fallback": global_params,
    }


def apply_calibration_params(frame: pd.DataFrame, params: dict, method_name: str) -> pd.DataFrame:
    """Apply calibration parameters to a prediction dataframe."""
    out = frame.copy()
    pred = out[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)

    if params["type"] == "none":
        out["calibration_method"] = "none"
        return out

    if params["type"] == "global_median":
        residuals = params["residuals"]
        adjustment = np.array([residuals["q25"], residuals["q50"], residuals["q75"]], dtype=float)
        calibrated = pred + adjustment[None, :]
        out[["pred_q25", "pred_q50", "pred_q75"]] = enforce_monotone_quantiles(calibrated)
        out["calibration_method"] = method_name
        return out

    if params["type"] == "binned_median":
        fallback = params["fallback"]["residuals"]
        calibrated = pred.copy()

        edges = params.get("edges")
        if edges is None:
            adjustment = np.array([fallback["q25"], fallback["q50"], fallback["q75"]], dtype=float)
            calibrated = pred + adjustment[None, :]
        else:
            edges = np.asarray(edges, dtype=float)
            bin_ids = assign_bins_from_edges(out[params["bin_feature"]], edges)
            pos_by_index = {idx: pos for pos, idx in enumerate(out.index)}
            for idx, bin_id in bin_ids.items():
                pos = pos_by_index[idx]
                if pd.isna(bin_id) or int(bin_id) < 0:
                    residual = fallback
                else:
                    residual = params["bins"].get(str(int(bin_id)), fallback)
                calibrated[pos, :] = pred[pos, :] + np.array(
                    [residual["q25"], residual["q50"], residual["q75"]],
                    dtype=float,
                )

        out[["pred_q25", "pred_q50", "pred_q75"]] = enforce_monotone_quantiles(calibrated)
        out["calibration_method"] = method_name
        return out

    raise ValueError(f"Unknown calibration type: {params['type']}")


def add_calibration_bin_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Add prediction-derived binning features used by calibration."""
    out = frame.copy()
    out["pred_iqr_for_calib"] = out["pred_q75"] - out["pred_q25"]
    out["pred_q75_for_calib"] = out["pred_q75"]
    return out

## 10. Fit and apply calibration

In [11]:
def calibrate_all_model_groups(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Fit and apply configured calibration methods per model group."""
    outputs = []
    parameter_rows = []

    data = add_calibration_bin_features(predictions)

    for keys, group in data.groupby(MODEL_ID_COLS, dropna=False):
        group = group.copy()
        group_id = dict(zip(MODEL_ID_COLS, keys))
        val = group.loc[group["split"].eq("val")].copy()

        none_output = group.copy()
        none_output["calibration_method"] = "none"
        outputs.append(none_output)
        parameter_rows.append({**group_id, "calibration_method": "none", "status": "identity", "n_val_rows": int(len(val))})

        if len(val) < cfg.min_val_rows_for_group_calibration:
            parameter_rows.append({**group_id, "calibration_method": "non_none", "status": "skipped_too_few_val_rows", "n_val_rows": int(len(val))})
            continue

        params = fit_global_median_calibration(val)
        outputs.append(apply_calibration_params(group, params, "global_median"))
        parameter_rows.append({
            **group_id, "calibration_method": "global_median", "status": "fit", "n_val_rows": int(len(val)),
            "residual_q25": params["residuals"]["q25"], "residual_q50": params["residuals"]["q50"], "residual_q75": params["residuals"]["q75"],
        })

        params = fit_binned_median_calibration(val, "pred_q75_for_calib", cfg.calibration_bins, cfg.min_bin_rows)
        outputs.append(apply_calibration_params(group, params, "pred_q75_decile"))
        parameter_rows.append({**group_id, "calibration_method": "pred_q75_decile", "status": "fit" if params.get("edges") is not None else "fallback_global", "n_val_rows": int(len(val)), "n_bins_fit": len(params.get("bins", {}))})

        params = fit_binned_median_calibration(val, "pred_iqr_for_calib", cfg.calibration_bins, cfg.min_bin_rows)
        outputs.append(apply_calibration_params(group, params, "pred_iqr_decile"))
        parameter_rows.append({**group_id, "calibration_method": "pred_iqr_decile", "status": "fit" if params.get("edges") is not None else "fallback_global", "n_val_rows": int(len(val)), "n_bins_fit": len(params.get("bins", {}))})

    calibrated = pd.concat(outputs, ignore_index=True)
    calibrated = add_prediction_error_columns(calibrated)
    calibrated = attach_segments(calibrated, test_segment_lookup)

    return calibrated, pd.DataFrame(parameter_rows)


calibrated_predictions, calibration_parameters = calibrate_all_model_groups(predictions)

safe_to_parquet(calibrated_predictions, output_dir / "calibrated_predictions_val_test.parquet")
calibration_parameters.to_csv(tables_dir / "calibration_parameters.csv", index=False)

print("Calibrated predictions:", calibrated_predictions.shape)
print_frame("Calibration parameter preview", calibration_parameters, max_rows=20)

Calibrated predictions: (534867, 45)

Calibration parameter preview
source                               model                  topology_candidate disruption_group checkpoint_metric           seed calibration_method   status  n_val_rows  residual_q25  residual_q50  residual_q75  n_bins_fit
ColdOD                   destination_prior                   destination_prior         baseline    not_applicable not_applicable               none identity         957           NaN           NaN           NaN         NaN
ColdOD                   destination_prior                   destination_prior         baseline    not_applicable not_applicable      global_median      fit         957     -9.140137      1.530029     40.974854         NaN
ColdOD                   destination_prior                   destination_prior         baseline    not_applicable not_applicable    pred_q75_decile      fit         957           NaN           NaN           NaN        10.0
ColdOD                   destination_pri

## 11. Calibration metrics and validation-selected calibration

In [12]:
CALIB_GROUP_COLS = MODEL_ID_COLS + ["calibration_method"]

calibration_metrics = []
for split in ["val", "test"]:
    metrics = compute_group_metrics(calibrated_predictions, CALIB_GROUP_COLS, split_filter=split)
    metrics["split"] = split
    calibration_metrics.append(metrics)

calibration_metrics = pd.concat(calibration_metrics, ignore_index=True)
calibration_metrics.to_csv(output_dir / "metrics_calibration_all_methods.csv", index=False)

val_metrics = calibration_metrics.loc[calibration_metrics["split"].eq("val")].copy()

if val_metrics.empty:
    selected_calibration_methods = (
        calibrated_predictions[MODEL_ID_COLS]
        .drop_duplicates()
        .assign(calibration_method="none", selection_reason="no_validation_rows", val_selected_pinball_mean=np.nan)
    )
else:
    selected_calibration_methods = (
        val_metrics.sort_values(MODEL_ID_COLS + ["pinball_mean"])
        .drop_duplicates(MODEL_ID_COLS, keep="first")
        [MODEL_ID_COLS + ["calibration_method", "pinball_mean"]]
        .rename(columns={"pinball_mean": "val_selected_pinball_mean"})
    )
    selected_calibration_methods["selection_reason"] = "min_val_pinball"

    all_groups = calibrated_predictions[MODEL_ID_COLS].drop_duplicates()
    selected_keys = selected_calibration_methods[MODEL_ID_COLS].drop_duplicates()
    missing_groups = all_groups.merge(selected_keys, on=MODEL_ID_COLS, how="left", indicator=True)
    missing_groups = missing_groups.loc[missing_groups["_merge"].eq("left_only"), MODEL_ID_COLS]

    if not missing_groups.empty:
        selected_calibration_methods = pd.concat(
            [
                selected_calibration_methods,
                missing_groups.assign(
                    calibration_method="none",
                    val_selected_pinball_mean=np.nan,
                    selection_reason="no_validation_rows",
                ),
            ],
            ignore_index=True,
        )

selected_calibration_methods.to_csv(tables_dir / "selected_calibration_methods_by_val.csv", index=False)

selected_calibrated_predictions = calibrated_predictions.merge(
    selected_calibration_methods[MODEL_ID_COLS + ["calibration_method"]],
    on=MODEL_ID_COLS + ["calibration_method"],
    how="inner",
    validate="many_to_one",
)

safe_to_parquet(selected_calibrated_predictions, output_dir / "selected_calibrated_predictions_val_test.parquet")

selected_calibration_metrics = []
for split in ["val", "test"]:
    metrics = compute_group_metrics(selected_calibrated_predictions, CALIB_GROUP_COLS, split_filter=split)
    metrics["split"] = split
    selected_calibration_metrics.append(metrics)

selected_calibration_metrics = pd.concat(selected_calibration_metrics, ignore_index=True)
selected_calibration_metrics.to_csv(output_dir / "metrics_selected_calibration.csv", index=False)

selected_test_leaderboard = (
    selected_calibration_metrics.loc[selected_calibration_metrics["split"].eq("test")]
    .sort_values("pinball_mean")
    .reset_index(drop=True)
)
selected_test_leaderboard.insert(0, "rank", np.arange(1, len(selected_test_leaderboard) + 1))
selected_test_leaderboard.to_csv(output_dir / "leaderboard_test_selected_calibration.csv", index=False)

print_frame("Selected calibration methods", selected_calibration_methods, max_rows=20)
print_frame("Selected calibration test leaderboard", selected_test_leaderboard, max_rows=20)


Selected calibration methods
source                               model                  topology_candidate disruption_group checkpoint_metric           seed calibration_method  val_selected_pinball_mean selection_reason
ColdOD                   destination_prior                   destination_prior         baseline    not_applicable not_applicable    pred_iqr_decile                 496.711609  min_val_pinball
ColdOD                   destination_prior                   destination_prior         baseline     postprocessed       baseline    pred_iqr_decile                 496.711607  min_val_pinball
ColdOD       fallback_lgbm_ensemble_global       fallback_lgbm_ensemble_global         baseline    not_applicable not_applicable               none                 208.912359  min_val_pinball
ColdOD       fallback_lgbm_ensemble_global       fallback_lgbm_ensemble_global         baseline     postprocessed       baseline               none                 208.912359  min_val_pinball
ColdOD fal

## 12. Calibrated segment summaries

In [13]:
calibrated_segment_tables = {}

for col in ["true_q75_decile", "true_iqr_decile", "n_fmi_county_pairs_quartile"]:
    table = segment_summary(selected_calibrated_predictions, col)
    calibrated_segment_tables[col] = table
    table.to_csv(tables_dir / f"calibrated_selected_segment_summary__{col}.csv", index=False)
    print_frame(f"Calibrated selected segment summary: {col}", table, max_rows=10)

true_q75_decile: all model groups sum to expected test rows = 1057.

Calibrated selected segment summary: true_q75_decile
source             model topology_candidate disruption_group checkpoint_metric           seed    true_q75_decile  n_rows  pinball_q25  pinball_q50  pinball_q75  pinball_mean  weighted_pinball_mean     mae_q25     mae_q50     mae_q75     iqr_mae     bias_q25     bias_q50     bias_q75  pred_iqr_mean  true_iqr_mean  raw_crossing_rate  q50_inside_q25_q75_rate
ColdOD destination_prior  destination_prior         baseline    not_applicable not_applicable true_q75_decile_01     106   974.872639   900.796404   663.198389    846.289144             841.463035 1299.830186 1801.592809 2652.793556 1352.963370  1299.830186  1801.592809  2652.793556    2197.315447     844.352077                0.0                 0.000000
ColdOD destination_prior  destination_prior         baseline    not_applicable not_applicable true_q75_decile_02     106   630.954748   503.022243   439.550347   

## 13. Decision-aware OD-risk CVaR screening evaluation

A full route-level CVaR regret evaluation requires candidate-route labels, which are not yet available for every alternative itinerary. This notebook implements a benchmark-compatible OD-risk screening proxy:

1. Compute predicted risk score: `pred_q75 + eta * max(pred_q75 - pred_q50, 0)`.
2. Compute realized risk score: `true_q75 + eta * max(true_q75 - true_q50, 0)`.
3. Select top-K highest-risk Cold-OD test lanes by predicted risk.
4. Compare realized risk captured by selected top-K lanes against the oracle top-K set chosen by true risk.

In [14]:
def add_cvar_risk_scores(frame: pd.DataFrame, eta: float) -> pd.DataFrame:
    """Add predicted and realized CVaR-like risk scores."""
    out = frame.copy()
    out["pred_iqr_upper"] = np.maximum(out["pred_q75"] - out["pred_q50"], 0.0)
    out["true_iqr_upper"] = np.maximum(out["true_q75"] - out["true_q50"], 0.0)
    out["pred_cvar_proxy"] = out["pred_q75"] + eta * out["pred_iqr_upper"]
    out["true_cvar_proxy"] = out["true_q75"] + eta * out["true_iqr_upper"]
    return out


def decision_scope_mask(test: pd.DataFrame, scope_name: str) -> pd.Series:
    """Return a boolean mask for a decision-evaluation scope."""
    if scope_name == "all_test":
        return pd.Series(True, index=test.index)
    if scope_name == "sparse_bottom25":
        return test["n_fmi_county_pairs_quartile"].astype(str).eq("n_fmi_q_01")
    if scope_name == "high_q75_top10":
        return test["true_q75_decile"].astype(str).eq("true_q75_decile_10")
    if scope_name == "high_iqr_top10":
        return test["true_iqr_decile"].astype(str).eq("true_iqr_decile_10")
    raise ValueError(f"Unknown decision scope: {scope_name}")


def evaluate_topk_risk_screening(
    frame: pd.DataFrame,
    group_cols: Sequence[str],
    eta: float,
    topk_fractions: Sequence[float],
    scopes: Sequence[str],
) -> pd.DataFrame:
    """Evaluate top-K OD-risk screening performance by model group."""
    data = add_cvar_risk_scores(frame, eta)
    test_all = data.loc[data["split"].astype(str).str.lower().eq("test")].copy()
    rows = []

    for keys, group in test_all.groupby(list(group_cols), dropna=False):
        group_id = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))

        for scope_name in scopes:
            scoped = group.loc[decision_scope_mask(group, scope_name)].copy()
            if scoped.empty:
                continue

            n = len(scoped)
            true_risk = scoped["true_cvar_proxy"]
            pred_risk = scoped["pred_cvar_proxy"]

            for frac in topk_fractions:
                k = int(math.ceil(frac * n))
                k = max(1, min(k, n))

                selected_idx = pred_risk.sort_values(ascending=False).head(k).index
                oracle_idx = true_risk.sort_values(ascending=False).head(k).index

                selected_realized_mean = float(true_risk.loc[selected_idx].mean())
                oracle_realized_mean = float(true_risk.loc[oracle_idx].mean())
                random_realized_mean = float(true_risk.mean())

                regret = oracle_realized_mean - selected_realized_mean
                normalized_regret = regret / max(abs(oracle_realized_mean), 1e-9)
                overlap = len(set(selected_idx) & set(oracle_idx)) / k

                rows.append({
                    **group_id,
                    "decision_scope": scope_name,
                    "topk_fraction": float(frac),
                    "n_scope_rows": int(n),
                    "k_selected": int(k),
                    "eta": float(eta),
                    "selected_realized_cvar_proxy_mean": selected_realized_mean,
                    "oracle_realized_cvar_proxy_mean": oracle_realized_mean,
                    "random_realized_cvar_proxy_mean": random_realized_mean,
                    "cvar_regret": float(regret),
                    "normalized_cvar_regret": float(normalized_regret),
                    "oracle_overlap_at_k": float(overlap),
                    "selected_mean_true_q75": float(scoped.loc[selected_idx, "true_q75"].mean()),
                    "oracle_mean_true_q75": float(scoped.loc[oracle_idx, "true_q75"].mean()),
                    "selected_mean_pred_cvar_proxy": float(pred_risk.loc[selected_idx].mean()),
                    "oracle_mean_pred_cvar_proxy": float(pred_risk.loc[oracle_idx].mean()),
                })

    return pd.DataFrame(rows)


decision_scopes = ["all_test", "sparse_bottom25", "high_q75_top10", "high_iqr_top10"]

decision_all_methods = evaluate_topk_risk_screening(
    calibrated_predictions,
    group_cols=CALIB_GROUP_COLS,
    eta=cfg.cvar_eta,
    topk_fractions=cfg.topk_fractions,
    scopes=decision_scopes,
)
decision_all_methods.to_csv(output_dir / "decision_cvar_risk_screening_all_calibrations.csv", index=False)

decision_selected = evaluate_topk_risk_screening(
    selected_calibrated_predictions,
    group_cols=CALIB_GROUP_COLS,
    eta=cfg.cvar_eta,
    topk_fractions=cfg.topk_fractions,
    scopes=decision_scopes,
)
decision_selected.to_csv(output_dir / "decision_cvar_risk_screening_selected_calibration.csv", index=False)

decision_leaderboard = (
    decision_selected
    .loc[
        decision_selected["decision_scope"].eq("all_test")
        & decision_selected["topk_fraction"].eq(0.10)
    ]
    .sort_values(["cvar_regret", "oracle_overlap_at_k"], ascending=[True, False])
    .reset_index(drop=True)
)
decision_leaderboard.insert(0, "rank", np.arange(1, len(decision_leaderboard) + 1))
decision_leaderboard.to_csv(output_dir / "decision_cvar_leaderboard_top10pct.csv", index=False)

print_frame("Decision CVaR leaderboard, top 10% all-test lanes", decision_leaderboard, max_rows=20)


Decision CVaR leaderboard, top 10% all-test lanes
 rank         source                             model                topology_candidate disruption_group      checkpoint_metric      seed calibration_method decision_scope  topk_fraction  n_scope_rows  k_selected  eta  selected_realized_cvar_proxy_mean  oracle_realized_cvar_proxy_mean  random_realized_cvar_proxy_mean  cvar_regret  normalized_cvar_regret  oracle_overlap_at_k  selected_mean_true_q75  oracle_mean_true_q75  selected_mean_pred_cvar_proxy  oracle_mean_pred_cvar_proxy
    1 GRAPH_ENSEMBLE graphsage_residual_prior_features graphsage_residual_prior_features         baseline           best_val_iqr      mean    pred_q75_decile       all_test            0.1          1057         106  0.5                        7801.807501                      8097.087129                      4645.013941   295.279629                0.036467             0.707547             6592.090659           6822.892362                    7826.378671           

## 14. Paired comparison after calibration

This cell compares each calibrated model against selected references on strict Cold-OD test rows.

In [15]:
def paired_summary_against_references(
    frame: pd.DataFrame,
    reference_names: Sequence[str],
    group_cols: Sequence[str],
) -> pd.DataFrame:
    """Compute row-level paired deltas against named reference models."""
    test = frame.loc[frame["split"].astype(str).str.lower().eq("test")].copy()
    rows = []

    for ref_name in reference_names:
        ref = test.loc[test["model"].astype(str).eq(ref_name)].copy()
        if ref.empty:
            continue

        ref = (
            ref.sort_values(["row_id", "calibration_method"])
            .drop_duplicates("row_id", keep="first")
            .copy()
        )
        ref_small = ref[
            [
                "row_id",
                "pinball_mean_row",
                "abs_error_q75",
                "abs_error_iqr",
            ]
        ].rename(
            columns={
                "pinball_mean_row": "ref_pinball_mean_row",
                "abs_error_q75": "ref_abs_error_q75",
                "abs_error_iqr": "ref_abs_error_iqr",
            }
        )

        for keys, cand in test.groupby(list(group_cols), dropna=False):
            cand = cand.drop_duplicates("row_id", keep="first").copy()
            merged = cand.merge(ref_small, on="row_id", how="inner", validate="one_to_one")
            if merged.empty:
                continue

            row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
            row.update({
                "reference_model": ref_name,
                "n_rows": int(len(merged)),
                "mean_delta_pinball": float((merged["ref_pinball_mean_row"] - merged["pinball_mean_row"]).mean()),
                "win_rate_pinball": float((merged["pinball_mean_row"] < merged["ref_pinball_mean_row"]).mean()),
                "mean_delta_q75_abs_error": float((merged["ref_abs_error_q75"] - merged["abs_error_q75"]).mean()),
                "win_rate_q75_abs_error": float((merged["abs_error_q75"] < merged["ref_abs_error_q75"]).mean()),
                "mean_delta_iqr_abs_error": float((merged["ref_abs_error_iqr"] - merged["abs_error_iqr"]).mean()),
                "win_rate_iqr_abs_error": float((merged["abs_error_iqr"] < merged["ref_abs_error_iqr"]).mean()),
            })
            rows.append(row)

    return pd.DataFrame(rows)


available_reference_names = [
    name for name in ["coldod_mlp_residual_prior", "graphv2_hgt", "graphv2_graphsage"]
    if name in set(selected_calibrated_predictions["model"].astype(str))
]

paired_calibrated = paired_summary_against_references(
    selected_calibrated_predictions,
    reference_names=available_reference_names,
    group_cols=CALIB_GROUP_COLS,
)

paired_calibrated.to_csv(output_dir / "paired_summary_selected_calibration_test_only.csv", index=False)
print_frame("Paired summary after selected calibration", paired_calibrated, max_rows=30)


Paired summary after selected calibration
                        source                                 model                    topology_candidate                disruption_group checkpoint_metric           seed calibration_method           reference_model  n_rows  mean_delta_pinball  win_rate_pinball  mean_delta_q75_abs_error  win_rate_q75_abs_error  mean_delta_iqr_abs_error  win_rate_iqr_abs_error
                        ColdOD                     destination_prior                     destination_prior                        baseline    not_applicable not_applicable    pred_iqr_decile coldod_mlp_residual_prior    1057         -310.942809          0.183538               -715.686034                0.263955               -232.231525                0.357616
                        ColdOD                     destination_prior                     destination_prior                        baseline     postprocessed       baseline    pred_iqr_decile coldod_mlp_residual_prior    1057       

## 15. Optional plots

In [16]:
if cfg.create_plots and MATPLOTLIB_AVAILABLE:
    plot_df = selected_test_leaderboard.head(25).copy()
    plot_df["label"] = (
        plot_df["source"].astype(str)
        + "::"
        + plot_df["model"].astype(str)
        + "::"
        + plot_df["calibration_method"].astype(str)
    )

    fig, ax = plt.subplots(figsize=(12, max(6, len(plot_df) * 0.35)))
    ax.barh(plot_df["label"][::-1], plot_df["pinball_mean"][::-1])
    ax.set_xlabel("pinball_mean")
    ax.set_title("Cold-OD Test Pinball Mean after Validation-Selected Calibration")
    fig.tight_layout()
    fig.savefig(plots_dir / "leaderboard_test_selected_calibration_pinball.png", dpi=180)
    plt.close(fig)

    if not decision_leaderboard.empty:
        plot_dec = decision_leaderboard.head(25).copy()
        plot_dec["label"] = (
            plot_dec["source"].astype(str)
            + "::"
            + plot_dec["model"].astype(str)
            + "::"
            + plot_dec["calibration_method"].astype(str)
        )
        fig, ax = plt.subplots(figsize=(12, max(6, len(plot_dec) * 0.35)))
        ax.barh(plot_dec["label"][::-1], plot_dec["cvar_regret"][::-1])
        ax.set_xlabel("CVaR screening regret, lower is better")
        ax.set_title("Decision-Aware OD-Risk Screening Regret, Top 10%")
        fig.tight_layout()
        fig.savefig(plots_dir / "decision_cvar_regret_top10pct.png", dpi=180)
        plt.close(fig)

    print("Plots saved to:", plots_dir)
else:
    print("Plot generation skipped.")

Plots saved to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\calibration_cvar_evaluation_v1_notebook\east_plus_gulf\plots


## 16. Write repaired report and artifact manifest

This report uses plain-text tables, not `to_markdown()`, so it does not require `tabulate`.

In [17]:
report_lines = []
report_lines.append("# Calibration and Decision-Aware CVaR Evaluation Report")
report_lines.append("")
report_lines.append(f"Created at Unix time: {time.time():.3f}")
report_lines.append(f"Output directory: {output_dir}")
report_lines.append("")
report_lines.append("## Experiment notes")
report_lines.append("- Sparse segments are rebuilt from the supervised row-level lookup.")
report_lines.append("- Reports use plain-text dataframe tables; no `tabulate` dependency is required.")
report_lines.append("- Calibration is fit on validation rows and selected by validation pinball.")
report_lines.append("- Decision-aware evaluation uses OD-lane CVaR-style risk screening.")
report_lines.append("")

report_lines.append("## Selected calibration test leaderboard")
report_lines.append(dataframe_to_text_table(
    selected_test_leaderboard,
    [
        "rank",
        "source",
        "model",
        "topology_candidate",
        "disruption_group",
        "checkpoint_metric",
        "seed",
        "calibration_method",
        "pinball_mean",
        "weighted_pinball_mean",
        "mae_q75",
        "iqr_mae",
    ],
    max_rows=25,
))
report_lines.append("")

report_lines.append("## Decision CVaR screening leaderboard, top 10% all-test lanes")
report_lines.append(dataframe_to_text_table(
    decision_leaderboard,
    [
        "rank",
        "source",
        "model",
        "topology_candidate",
        "disruption_group",
        "checkpoint_metric",
        "seed",
        "calibration_method",
        "cvar_regret",
        "normalized_cvar_regret",
        "oracle_overlap_at_k",
        "selected_realized_cvar_proxy_mean",
        "oracle_realized_cvar_proxy_mean",
    ],
    max_rows=25,
))
report_lines.append("")

report_lines.append("## Sparse segment audit")
report_lines.append(dataframe_to_text_table(sparse_audit, max_rows=30))
report_lines.append("")

report_lines.append("## Paired summary after selected calibration")
report_lines.append(dataframe_to_text_table(
    paired_calibrated,
    [
        "reference_model",
        "source",
        "model",
        "topology_candidate",
        "disruption_group",
        "checkpoint_metric",
        "seed",
        "calibration_method",
        "n_rows",
        "mean_delta_pinball",
        "win_rate_pinball",
        "mean_delta_q75_abs_error",
        "win_rate_q75_abs_error",
    ],
    max_rows=30,
))

report_path = reports_dir / "calibration_cvar_evaluation_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

artifact_manifest = {
    "output_dir": str(output_dir),
    "normalized_predictions_with_repaired_segments": str(output_dir / "normalized_predictions_with_repaired_segments.parquet"),
    "calibrated_predictions": str(output_dir / "calibrated_predictions_val_test.parquet"),
    "selected_calibrated_predictions": str(output_dir / "selected_calibrated_predictions_val_test.parquet"),
    "metrics_calibration_all_methods": str(output_dir / "metrics_calibration_all_methods.csv"),
    "metrics_selected_calibration": str(output_dir / "metrics_selected_calibration.csv"),
    "leaderboard_test_selected_calibration": str(output_dir / "leaderboard_test_selected_calibration.csv"),
    "decision_cvar_all_calibrations": str(output_dir / "decision_cvar_risk_screening_all_calibrations.csv"),
    "decision_cvar_selected_calibration": str(output_dir / "decision_cvar_risk_screening_selected_calibration.csv"),
    "decision_cvar_leaderboard_top10pct": str(output_dir / "decision_cvar_leaderboard_top10pct.csv"),
    "paired_summary_selected_calibration_test_only": str(output_dir / "paired_summary_selected_calibration_test_only.csv"),
    "tables_dir": str(tables_dir),
    "plots_dir": str(plots_dir),
    "report": str(report_path),
}

safe_write_json(artifact_manifest, output_dir / "analysis_artifact_manifest_calibration_cvar.json")

run_config = {
    "notebook": "Calibration_and_CVaR_Evaluation",
    "config": asdict(cfg),
    "paths": {
        "supervised_path": str(supervised_path),
        "disruption_predictions_path": str(disruption_predictions_path),
        "graphv2_predictions_path": str(graphv2_predictions_path),
        "coldod_predictions_path": str(coldod_predictions_path),
        "output_dir": str(output_dir),
    },
    "created_at_unix": time.time(),
    "package_versions": {
        "python": os.sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "matplotlib_available": MATPLOTLIB_AVAILABLE,
    },
}

safe_write_json(run_config, output_dir / "run_config_calibration_cvar.json")

print("Report saved to:", report_path)
print("Artifact manifest:")
print(json.dumps(artifact_manifest, indent=2))

Report saved to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\calibration_cvar_evaluation_v1_notebook\east_plus_gulf\reports\calibration_cvar_evaluation_report.md
Artifact manifest:
{
  "output_dir": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\calibration_cvar_evaluation_v1_notebook\\east_plus_gulf",
  "normalized_predictions_with_repaired_segments": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\calibration_cvar_evaluation_v1_notebook\\east_plus_gulf\\normalized_predictions_with_repaired_segments.parquet",
  "calibrated_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\calibration_cvar_evaluation_v1_notebook\\east_plus_gulf\\calibrated_predictions_val_test.parquet",
  "selected_calibrated_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\calibration_cvar_evaluation_v1_notebook\\east_plus_gulf\\selected_calibrated_predictions_val_test.parquet",
  "metrics_calibration_all_methods": 

## 17. Interpretation checklist

After running the notebook, inspect these files first:

```text
leaderboard_test_selected_calibration.csv
metrics_selected_calibration.csv
paired_summary_selected_calibration_test_only.csv
decision_cvar_leaderboard_top10pct.csv
tables/repaired_sparse_segment_audit.csv
tables/calibrated_selected_segment_summary__n_fmi_county_pairs_quartile.csv
reports/calibration_cvar_evaluation_report.md
```

Key questions:

1. Does calibration improve test pinball without worsening q75/IQR?
2. Does calibration improve high-q75 or high-IQR segment metrics?
3. Do repaired sparse summaries now show `n_fmi_q_01` through `n_fmi_q_04` for every model group?
4. Which model has the lowest top-10% OD-risk screening regret?
5. Does the weather-gated model identify high-risk OD lanes better than the no-disruption and GraphV2 references?